<a href="https://colab.research.google.com/github/cyanm002-maker/Cascade/blob/main/Cascade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [1]:
import os
import shutil
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


# This part is for the pytorch and cuda

In [2]:
from torch.cuda import is_available
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("Pytorch version:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:",torch.cuda.get_device_name(0))

Pytorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


# Inspect the dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive')

src_path = "/content/drive/MyDrive/Cascade/dataset_2_1 (1).hdf5"
dst_path = "/content/dataset_2_1.hdf5"
shutil.copy(src_path, dst_path)

if not os.path.exists(dst_path):
  shutil.copy(src_path, dst_path)

with h5py.File(dst_path, 'r') as f:
  print("HDF5 keys:", list(f.keys()))
  for k in f.keys():
      print(f"{k}: shape={f[k].shape}, dtype={f[k].dtype}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HDF5 keys: ['incident_energies', 'showers']
incident_energies: shape=(100000, 1), dtype=float64
showers: shape=(100000, 6480), dtype=float64


# Transform,DataSet and batch check

In [4]:
def encode_energy(energy_gev):
  return np.log10(np.clip(energy_gev, 1e-10, None))


def preprocess_voxels(voxel_mev, eps=1e-6):
  clean = np.clip(voxel_mev.astype(np.float32), 0.0, None)
  return np.log(clean + eps)

def invert_voxels(pred_log, eps=1e-6, threshold_mev = 0.015):
  x = np.exp(pred_log) - eps
  x = np.clip(x, 0.0, None)
  return np.where(x > threshold_mev, x, 0.0)

class ShowerForgeDataset(Dataset):
  def __init__(self, hdf5_path, shower_key, energy_key, max_events=None, eps=1e-6):
    self.hdf5_path = hdf5_path
    self.shower_key = shower_key
    self.energy_key = energy_key
    self.eps = eps
    self.max_events = max_events
    with h5py.File(self.hdf5_path, 'r') as f:
      total = f[self.energy_key].shape[0]
      self.length = min(total, max_events) if max_events is not None else total
      self.energies = f[self.energy_key][: self.length].astype(np.float32).squeeze()

  def __len__(self):
    return self.length

  def __getitem__(self, index):
    with h5py.File(self.hdf5_path, "r") as f:
        voxel = f[self.shower_key][index].astype(np.float32)
    x_log = preprocess_voxels(voxel, eps=self.eps)
    c_log = encode_energy(self.energies[index])
    return torch.from_numpy(x_log).float(), torch.tensor([c_log], dtype=torch.float32)

# testing on 5k shower
ds = ShowerForgeDataset("/content/dataset_2_1.hdf5", "showers", "incident_energies", max_events=5_000)
loader = DataLoader(ds, batch_size=64, shuffle=True)

x_batch, c_batch = next(iter(loader))
print("x_batch:", x_batch.shape, "c_batch:", c_batch.shape)

x_batch: torch.Size([64, 6480]) c_batch: torch.Size([64, 1])


# Conditional VAE implimentation

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [6]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure src directory exists
os.makedirs("src", exist_ok=True)

class ConditionalVAE(nn.Module):
  def __init__(self, input_dim: int = 6480, cond_dim: int = 1, latent_dim: int = 64, hidden_dim: int = 512):
    super().__init__()
    self.input_dim = input_dim
    self.cond_dim = cond_dim
    self.latent_dim = latent_dim

    # Encoder: [x_flat, cond] -> (mu, logvar)
    self.enc = nn.Sequential(
        nn.Linear(input_dim + cond_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.SiLU()
    )

    self.fc_mu = nn.Linear(hidden_dim, latent_dim)
    self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    # Decoder: [z, cond] -> x
    self.dec = nn.Sequential(
        nn.Linear(latent_dim + cond_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, input_dim)
    )

  def encode(self, x: torch.Tensor, c: torch.Tensor):
    x_flat = x.view(x.size(0), -1)
    h = self.enc(torch.cat([x_flat, c], dim=-1))
    return self.fc_mu(h), self.fc_logvar(h)

  @staticmethod
  def reparameterize(mu: torch.Tensor, logvar: torch.Tensor):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

  def decode(self, z: torch.Tensor, c: torch.Tensor, out_shape: tuple):
    out = self.dec(torch.cat([z, c], dim=-1))
    return out.view(out_shape)

  def forward(self, x: torch.Tensor, c: torch.Tensor):
    mu, logvar = self.encode(x, c)
    z = self.reparameterize(mu, logvar)
    recon = self.decode(z, c, x.shape)
    return recon, mu, logvar

  @torch.no_grad()
  def sample(self, cond: torch.Tensor, shower_shape: tuple = (45, 16, 9), device: str = "cuda"):
    """Generate showers given condition tensor of shape (B, cond_dim)."""
    batch_size = cond.shape[0]
    z = torch.randn(batch_size, self.latent_dim, device=device)
    cond = cond.to(device)
    return self.decode(z, cond, (batch_size, *shower_shape))

def cvae_loss(recon: torch.Tensor, target: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor, beta: float = 0.01):
  recon_loss = F.mse_loss(recon, target, reduction="mean")
  kl_loss = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())
  return recon_loss + beta * kl_loss, recon_loss.detach(), kl_loss.detach()

# Save implementation code to src/models.py
models_code = """import torch
import torch.nn as nn
import torch.nn.functional as F

class ConditionalVAE(nn.Module):
  def __init__(self, input_dim: int = 6480, cond_dim: int = 1, latent_dim: int = 64, hidden_dim: int = 512):
    super().__init__()
    self.input_dim = input_dim
    self.cond_dim = cond_dim
    self.latent_dim = latent_dim

    self.enc = nn.Sequential(
        nn.Linear(input_dim + cond_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.SiLU()
    )

    self.fc_mu = nn.Linear(hidden_dim, latent_dim)
    self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    self.dec = nn.Sequential(
        nn.Linear(latent_dim + cond_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, input_dim)
    )

  def encode(self, x: torch.Tensor, c: torch.Tensor):
    x_flat = x.view(x.size(0), -1)
    h = self.enc(torch.cat([x_flat, c], dim=-1))
    return self.fc_mu(h), self.fc_logvar(h)

  @staticmethod
  def reparameterize(mu: torch.Tensor, logvar: torch.Tensor):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

  def decode(self, z: torch.Tensor, c: torch.Tensor, out_shape: tuple):
    out = self.dec(torch.cat([z, c], dim=-1))
    return out.view(out_shape)

  def forward(self, x: torch.Tensor, c: torch.Tensor):
    mu, logvar = self.encode(x, c)
    z = self.reparameterize(mu, logvar)
    recon = self.decode(z, c, x.shape)
    return recon, mu, logvar

  @torch.no_grad()
  def sample(self, cond: torch.Tensor, shower_shape: tuple = (45, 16, 9), device: str = "cuda"):
    batch_size = cond.shape[0]
    z = torch.randn(batch_size, self.latent_dim, device=device)
    cond = cond.to(device)
    return self.decode(z, cond, (batch_size, *shower_shape))

def cvae_loss(recon: torch.Tensor, target: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor, beta: float = 0.01):
  recon_loss = F.mse_loss(recon, target, reduction="mean")
  kl_loss = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())
  return recon_loss + beta * kl_loss, recon_loss.detach(), kl_loss.detach()
"""

with open("src/models.py", "w") as f:
  f.write(models_code)

if os.getcwd() not in sys.path:
  sys.path.insert(0, os.getcwd())

print("ConditionalVAE and cvae_loss written to src/models.py successfully!")

ConditionalVAE and cvae_loss written to src/models.py successfully!


# traning loop with CheckPoints

In [7]:
import os
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

# We will use ShowerForgeDataset defined in the notebook instead of importing from src.dataset
from src.models import ConditionalVAE, cvae_loss


def train_cvae(
    hdf5_path: str = "/content/dataset_2_1.hdf5",
    shower_key: str = "showers",
    energy_key: str = "incident_energies",
    ckpt_dir: str = "/content/drive/MyDrive/ShowerForge/outputs/checkpoints",
    max_events: int | None = 10_000,
    batch_size: int = 128,
    epochs: int = 10,
    lr: float = 1e-3,
    beta: float = 0.01,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    os.makedirs(ckpt_dir, exist_ok=True)

    # Using ShowerForgeDataset from the current notebook workspace
    dataset = ShowerForgeDataset(
        hdf5_path=hdf5_path,
        shower_key=shower_key,
        energy_key=energy_key,
        max_events=max_events,
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    sample_x, sample_c = dataset[0]
    input_dim = sample_x.numel()
    cond_dim = sample_c.numel()

    model = ConditionalVAE(input_dim=input_dim, cond_dim=cond_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_rec, total_kl = 0.0, 0.0, 0.0

        for x, c in tqdm(loader, desc=f"Epoch {epoch}/{epochs}", leave=False):
            x, c = x.to(device), c.to(device)
            optimizer.zero_grad()

            recon, mu, logvar = model(x, c)
            loss, rec, kl = cvae_loss(recon, x, mu, logvar, beta=beta)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_rec += rec.item()
            total_kl += kl.item()

        n_batches = len(loader)
        print(
            f"Epoch {epoch:02d} | Loss: {total_loss/n_batches:.4f} | "
            f"Recon: {total_rec/n_batches:.4f} | KL: {total_kl/n_batches:.4f}"
        )

        ckpt_path = os.path.join(ckpt_dir, f"cvae_epoch_{epoch:02d}.pt")
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            ckpt_path,
        )

# Load checkpoint to show the shower

In [8]:
import numpy as np
import torch

def encode_energy(energy_gev):
  return np.log10(np.clip(energy_gev, 1e-10, None))

def invert_voxels(pred_log, eps=1e-6, threshold_mev = 0.015):
  x = np.exp(pred_log) - eps
  x = np.clip(x, 0.0, None)
  return np.where(x>threshold_mev,x,0.0)

@torch.no_grad()
def generate_showers(
    model,
    ckpt_path:str,
    energies_gev:np.ndarray,
    batch_size:int = 128,
    shower_shape:tuple = (45,16,9),
    eps: float = 1e-6,
    threshold_mev:float = 0.015,
    device:str = "cuda",
):
  ckpt = torch.load(ckpt_path,map_location=device)
  state_dict = ckpt.get("model_state_dict") or ckpt.get("model_state") or ckpt
  model.load_state_dict(state_dict)
  model.to(device)
  model.eval()

  cond_all = encode_energy(energies_gev.astype(np.float32)).reshape(-1,1)
  generated_mev = []

  for start in range(0,len(cond_all),batch_size):
    c_batch = torch.tensor(cond_all[start:start+batch_size],dtype=torch.float32,device=device)
    pred_log = model.sample(c_batch,shower_shape=shower_shape,device=device).cpu().numpy()
    pred_mev = invert_voxels(pred_log,eps=eps,threshold_mev=threshold_mev)
    generated_mev.append(pred_mev)


  return np.concatenate(generated_mev,axis=0)

# Compute First Physics Observables

In [9]:
import numpy as np

def compute_physics_observables(showers_mev: np.ndarray, energies_gev: np.ndarray):
    """
    showers_mev: array of shape (N, 45, 16, 9) in MeV (after readout threshold)
    energies_gev: array of shape (N,) in GeV (1 GeV = 1000 MeV)
    """
    energies_mev = energies_gev.reshape(-1) * 1000.0
    total_energy = showers_mev.sum(axis=(1, 2, 3))
    energy_ratio = total_energy / np.clip(energies_mev, 1e-8, None)

    longitudinal_profile = showers_mev.sum(axis=(2, 3)).mean(axis=0)  # shape (45,)
    angular_profile = showers_mev.sum(axis=(1, 3)).mean(axis=0)       # shape (16,)
    radial_profile = showers_mev.sum(axis=(1, 2)).mean(axis=0)        # shape (9,)
    sparsity = np.mean(showers_mev == 0.0, axis=(1, 2, 3))           # shape (N,)

    return {
        "total_energy": total_energy,
        "energy_ratio": energy_ratio,
        "longitudinal_profile": longitudinal_profile,
        "angular_profile": angular_profile,
        "radial_profile": radial_profile,
        "sparsity": sparsity,
    }